# Sequence-Aware Recommendation Feeds with SSD

Most diversification strategies — DPP, MMR, COVER — treat each request as independent.
They diversify **within** a single batch but have no memory across requests.
A user who scrolls through several pages of results will keep seeing the same dominant topics recycled page after page.

**SSD (Sliding Spectrum Decomposition)** is different.
It diversifies **across** pages by penalising items that are similar to what was recently shown.
Each new page fills in topic gaps left by previous ones, producing a feed that feels genuinely fresh.

---

**Scenario:** A news app learns from reading history that this user loves Apple/tech coverage. The ranking model assigns high relevance scores to Apple articles, medium scores to other tech, and lower scores to world and sports news. Without cross-page memory (DPP), Apple articles dominate every page. With SSD's rolling history, the feed gradually broadens as Apple news is "used up" — surfacing other tech stories, then world and sports.

**Corpus:** 240 AG News articles — 80 Apple-related tech, 80 other tech, 80 world/sports.

In [1]:
%pip install pyversity sentence-transformers datasets


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pyversity import diversify, Strategy

/Users/thomasvandongen/.pyenv/versions/3.12.8/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The Corpus

We build a corpus from AG News with three groups:

| Topic | Description | Articles |
|-------|-------------|----------|
| **Apple** | Sci/Tech articles mentioning Apple | 80 |
| **General Tech** | Other Sci/Tech articles | 80 |
| **World & Sports** | World and Sports articles | 80 |

The dominant group (Apple) is the one the ranking model has learned to score highest for this user.

In [3]:
TOPIC_NAMES  = {0: "Apple", 1: "General Tech", 2: "World & Sports"}
TARGET_COUNT = 80

ds = load_dataset("ag_news", split="train")

apple        = []
general_tech = []
world_sports = []

for row in ds:
    text, label = row["text"], row["label"]
    if label == 3 and "apple" in text.lower() and len(apple) < TARGET_COUNT:
        apple.append(text)
    elif label == 3 and len(general_tech) < TARGET_COUNT:
        general_tech.append(text)
    elif label in (0, 1) and len(world_sports) < TARGET_COUNT:
        world_sports.append(text)
    if len(apple) >= TARGET_COUNT and len(general_tech) >= TARGET_COUNT and len(world_sports) >= TARGET_COUNT:
        break

texts  = apple + general_tech + world_sports
topics = [0]*len(apple) + [1]*len(general_tech) + [2]*len(world_sports)

print(f"Corpus size: {len(texts)} articles")
for t, name in TOPIC_NAMES.items():
    print(f"  {name:18s}: {topics.count(t)} articles")

Corpus size: 240 articles
  Apple             : 80 articles
  General Tech      : 80 articles
  World & Sports    : 80 articles


## Encoding

We encode each article with [**potion-base-32M**](https://huggingface.co/minishlab/potion-base-32M),
a fast static embedding model from the [model2vec](https://github.com/MinishLab/model2vec) family.
These embeddings are used **only for diversity computation** — they tell each strategy how similar
two articles are so it can avoid redundancy.

In [4]:
model = SentenceTransformer("minishlab/potion-base-32M", device="cpu")

embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print(f"Embedding matrix: {embeddings.shape}")

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches: 100%|██████████| 8/8 [00:00<00:00, 283.94it/s]

Embedding matrix: (240, 512)


## Relevance Scores

In a real system, the ranking model would output a score per article.
Here we simulate that directly: the user's reading history has taught the model
to rank Apple articles highest, other tech second, and world/sports third.
Small random noise is added to reflect natural score variability.

In [5]:
# Learned user preference scores (Apple-heavy)
PREFERENCE = {0: 0.75, 1: 0.65, 2: 0.55}  # Apple, General Tech, World & Sports

np.random.seed(42)
relevance_scores = np.array([
    PREFERENCE[t] + np.random.uniform(-0.02, 0.02)
    for t in topics
])

print("Relevance score distribution:")
for t, name in TOPIC_NAMES.items():
    s = relevance_scores[[i for i, topic in enumerate(topics) if topic == t]]
    print(f"  {name:18s}: mean={s.mean():.3f}  (range {s.min():.3f}–{s.max():.3f})")

Relevance score distribution:
  Apple             : mean=0.749  (range 0.730–0.769)
  General Tech      : mean=0.650  (range 0.630–0.669)
  World & Sports    : mean=0.549  (range 0.530–0.569)


---
## Simulating a Multi-Page Feed

The user scrolls through 4 pages of 8 articles each.
Both feeds exclude already-shown articles from subsequent pages.

| Feed | Strategy | History |
|------|----------|---------|
| **DPP feed** | DPP, `diversity=0.95` | None — each page diversifies within itself |
| **SSD feed** | SSD, `diversity=0.95` | Rolling — each page penalises topics from previous pages |

Both use the same `diversity` parameter — the only difference is whether SSD's rolling history is active.

In [6]:
N_PAGES   = 4
PAGE_SIZE = 8
DIVERSITY = 0.95

# ── DPP feed (no history) ───────────────────────────────────────────────────
dpp_shown = set()
dpp_pages = []

for page in range(N_PAGES):
    available   = [i for i in range(len(texts)) if i not in dpp_shown]
    result      = diversify(embeddings[available], relevance_scores[available],
                            k=PAGE_SIZE, strategy=Strategy.DPP, diversity=DIVERSITY)
    selected    = [available[i] for i in result.indices]
    dpp_pages.append(selected)
    dpp_shown.update(selected)

# ── SSD feed (rolling history) ──────────────────────────────────────────────
ssd_shown   = set()
ssd_pages   = []
ssd_history = None

for page in range(N_PAGES):
    available   = [i for i in range(len(texts)) if i not in ssd_shown]
    kwargs      = {"recent_embeddings": ssd_history} if ssd_history is not None else {}
    result      = diversify(embeddings[available], relevance_scores[available],
                            k=PAGE_SIZE, strategy=Strategy.SSD, diversity=DIVERSITY, **kwargs)
    selected    = [available[i] for i in result.indices]
    ssd_pages.append(selected)
    ssd_shown.update(selected)

    shown_emb   = embeddings[selected]
    ssd_history = shown_emb if ssd_history is None else np.vstack([ssd_history, shown_emb])

print("Simulation complete.")

Simulation complete.


---
## Topic Coverage: DPP vs SSD

For each page we count how many articles fell into each topic.
The **Topics covered** row tracks the cumulative count of distinct topics reached after each page.

In [7]:
topic_order = list(TOPIC_NAMES.values())

def coverage_table(pages, label):
    counts = {name: [0]*len(pages) for name in topic_order}
    for p_idx, page in enumerate(pages):
        for i in page:
            counts[TOPIC_NAMES[topics[i]]][p_idx] += 1

    col_w  = 9
    header = f"{'Topic':<20}" + "".join(f"{'Page '+str(p+1):>{col_w}}" for p in range(len(pages)))
    sep    = '─' * len(header)
    print(f"\n{sep}\n  {label}\n{sep}\n{header}\n{sep}")

    topics_per_page = [set() for _ in pages]
    for name in topic_order:
        row = f"{name:<20}"
        for p_idx, cnt in enumerate(counts[name]):
            row += f"{(str(cnt) if cnt > 0 else '·'):>{col_w}}"
            if cnt > 0:
                topics_per_page[p_idx].add(name)
        print(row)

    print(sep)
    cumulative = set()
    cumul_row  = f"{'Topics covered':<20}"
    for p_idx in range(len(pages)):
        cumulative.update(topics_per_page[p_idx])
        cumul_row += f"{len(cumulative):>{col_w}}"
    print(cumul_row)
    print(sep)

coverage_table(dpp_pages, "DPP  (no history — Apple dominates every page)")
coverage_table(ssd_pages, "SSD  (rolling history — broadens over time)")


────────────────────────────────────────────────────────
  DPP  (no history — Apple dominates every page)
────────────────────────────────────────────────────────
Topic                  Page 1   Page 2   Page 3   Page 4
────────────────────────────────────────────────────────
Apple                       5        5        5        4
General Tech                3        3        3        4
World & Sports              ·        ·        ·        ·
────────────────────────────────────────────────────────
Topics covered              2        2        2        2
────────────────────────────────────────────────────────

────────────────────────────────────────────────────────
  SSD  (rolling history — broadens over time)
────────────────────────────────────────────────────────
Topic                  Page 1   Page 2   Page 3   Page 4
────────────────────────────────────────────────────────
Apple                       5        3        2        1
General Tech                2        3        4 

---
## What the Feed Looks Like

Reading the actual headlines makes the difference tangible.

In [8]:
def print_feed(pages, label):
    print(f"\n{'═'*80}\n  {label}\n{'═'*80}")
    for p_idx, page in enumerate(pages):
        print(f"\n  ── Page {p_idx+1} ──")
        for rank, i in enumerate(page, 1):
            topic = TOPIC_NAMES[topics[i]]
            print(f"    {rank}. [{topic:<14}]  {texts[i][:68]}")

print_feed(dpp_pages, "DPP feed (no history)")
print_feed(ssd_pages, "SSD feed (rolling history)")


════════════════════════════════════════════════════════════════════════════════
  DPP feed (no history)
════════════════════════════════════════════════════════════════════════════════

  ── Page 1 ──
    1. [Apple         ]  Apple recalls 28,000 PowerBook batteries Apple Computer Inc. Thursda
    2. [Apple         ]  Apple Introduces Production Suite Production Suite, essential softwa
    3. [Apple         ]  Real v Apple music war: iPod freedom petition backfires Hostilities 
    4. [Apple         ]  Analysts concerned about longer-than-expected G5 delays AUGUST 17, 2
    5. [General Tech  ]  Rescuers Free Beached Whale in Brazil (AP) AP - Rescuers succeeded i
    6. [General Tech  ]  Natural Sunblock: Sun Dims in Strange Ways (SPACE.com) SPACE.com - W
    7. [General Tech  ]  Scientists Probe Pacific for Dead Zone (AP) AP - His hand on a toggl
    8. [Apple         ]  Hacker takes bite out of Apple's iTunes The Norwegian hacker famous 

  ── Page 2 ──
    1. [Apple         ]  Riva

---
## Key Takeaways

1. **DPP diversifies within a page, not across pages.**
   Each page is internally diverse, but the highest-scoring topic (Apple) dominates every page.
   World & Sports never appears — users who scroll 4 pages see no variety beyond tech news.

2. **SSD diversifies across pages using a rolling history.**
   Pass `recent_embeddings` — the stacked embeddings of previously shown items —
   and SSD penalises candidates similar to that history.
   Apple gradually gives way to General Tech and then World & Sports.

3. **Growing the history window is one line.**
   ```python
   ssd_history = shown_emb if ssd_history is None else np.vstack([ssd_history, shown_emb])
   ```
   In production, cap the window to the last `k` items to bound memory
   and prevent over-penalising older topics:
   ```python
   ssd_history = np.vstack([ssd_history, shown_emb])[-PAGE_SIZE:]
   ```

4. **The `diversity` parameter works the same as for all other strategies.**
   `0.0` = pure relevance, `1.0` = maximum novelty.
   Values of `0.8–0.95` work well for recommendation feeds.